# Pendulum Length Estimation from IMU Data

An IMU was mounted in a bucket suspended from a pendulum. We analyze the accelerometer and gyroscope data to:
1. Visualize the raw IMU signals
2. Identify a clean swinging window (accounting for rotation of the bucket)
3. Extract the pendulum period *T*
4. Estimate the pendulum length using $L = g\left(\frac{T}{2\pi}\right)^2$

**Challenge:** The bucket rotated during the experiment, so the horizontal swing signal is spread across both AccX and AccY. We handle this using two methods:
- **PCA projection** – find the dominant horizontal oscillation direction and project onto it for a signed, rotation-robust signal
- **Horizontal magnitude** – $|a_H| = \sqrt{\text{AccX}^2 + \text{AccY}^2}$, which captures swing regardless of orientation (but oscillates at *twice* the pendulum frequency)

**Data files:** `imuLogPendulumTest1.csv` (≈130 s) and `imuLogPendulumTest2.csv` (≈335 s)

## 1. Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal
from scipy.fft import rfft, rfftfreq
from datetime import datetime

g = 9.81  # m/s²

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

## 2. Load Data

In [ ]:
def load_imu(filepath):
    """Load an IMU CSV file and return a tidy DataFrame.

    Accelerations are converted from milli-g to m/s².
    Time is expressed in seconds elapsed from the first sample.
    """
    df = pd.read_csv(filepath)
    df['time'] = pd.to_datetime(df['Timestamp'], format='%Y/%m/%d %H:%M:%S.%f')
    t0 = df['time'].iloc[0]
    df['t_s'] = (df['time'] - t0).dt.total_seconds()

    # Convert accelerations: milli-g → m/s²
    for ax in ['AccX', 'AccY', 'AccZ']:
        df[ax] = df[ax] / 1000.0 * g

    # Horizontal acceleration magnitude (always positive)
    df['AccH_mag'] = np.sqrt(df['AccX']**2 + df['AccY']**2)

    # Estimate sample rate
    dt = np.median(np.diff(df['t_s'].values))
    fs = 1.0 / dt

    print(f"Loaded {filepath}")
    print(f"  Samples : {len(df)}")
    print(f"  Duration: {df['t_s'].iloc[-1]:.1f} s")
    print(f"  Sample rate: {fs:.2f} Hz  (median dt = {dt*1000:.1f} ms)")
    print()
    return df, fs


df1, fs1 = load_imu('imuLogPendulumTest1.csv')
df2, fs2 = load_imu('imuLogPendulumTest2.csv')

## 3. Raw Signal Overview

In [ ]:
def plot_overview(df, fs, title):
    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

    axes[0].plot(df['t_s'], df['AccX'], label='AccX', alpha=0.8)
    axes[0].plot(df['t_s'], df['AccY'], label='AccY', alpha=0.8)
    axes[0].plot(df['t_s'], df['AccZ'], label='AccZ', alpha=0.6)
    axes[0].set_ylabel('Acceleration (m/s²)')
    axes[0].set_title(f'{title} – Accelerometer')
    axes[0].legend(loc='upper right')
    axes[0].grid(True, alpha=0.4)

    axes[1].plot(df['t_s'], df['GyrX'], label='GyrX', alpha=0.8)
    axes[1].plot(df['t_s'], df['GyrY'], label='GyrY', alpha=0.8)
    axes[1].plot(df['t_s'], df['GyrZ'], label='GyrZ (spin)', color='red', alpha=0.9)
    axes[1].set_ylabel('Angular rate (deg/s)')
    axes[1].set_title('Gyroscope  (GyrZ = bucket spin about vertical)')
    axes[1].legend(loc='upper right')
    axes[1].grid(True, alpha=0.4)

    axes[2].plot(df['t_s'], df['AccH_mag'], color='purple', label='|AccH|', alpha=0.9)
    axes[2].set_ylabel('|AccH| (m/s²)')
    axes[2].set_xlabel('Time (s)')
    axes[2].set_title('Horizontal acceleration magnitude')
    axes[2].legend(loc='upper right')
    axes[2].grid(True, alpha=0.4)

    fig.tight_layout()
    plt.show()


plot_overview(df1, fs1, 'Test 1')
plot_overview(df2, fs2, 'Test 2')

## 4. Identify the Swinging Window

We look for a time window where:
- The pendulum is clearly swinging: horizontal acceleration magnitude is large and oscillatory
- Bucket rotation is modest: |GyrZ| is small relative to the pendulum's angular rate

We compute a short-time RMS of `AccH_mag` and `GyrZ` to find this window automatically.

In [ ]:
def compute_rms_envelopes(df, fs, rms_window_s=2.0):
    """Compute short-time RMS of |AccH| and |GyrZ| for plotting."""
    win   = max(1, int(rms_window_s * fs))
    box   = np.ones(win) / win
    rms_h  = np.sqrt(np.convolve(df['AccH_mag'].values**2,          box, mode='same'))
    rms_gz = np.sqrt(np.convolve(np.abs(df['GyrZ'].values)**2,      box, mode='same'))
    return rms_h, rms_gz


# ── Manual window (seconds) ──────────────────────────────────────────────────
T_WIN_START = 50.0
T_WIN_END   = 110.0
# ─────────────────────────────────────────────────────────────────────────────

rms_h1, rms_gz1 = compute_rms_envelopes(df1, fs1)
rms_h2, rms_gz2 = compute_rms_envelopes(df2, fs2)

t1_start, t1_end = T_WIN_START, min(T_WIN_END, df1['t_s'].iloc[-1])
t2_start, t2_end = T_WIN_START, min(T_WIN_END, df2['t_s'].iloc[-1])

print(f"Test 1 window: {t1_start:.1f} – {t1_end:.1f} s")
print(f"Test 2 window: {t2_start:.1f} – {t2_end:.1f} s")

In [ ]:
def plot_window_selection(df, t_start, t_end, rms_h, rms_gz, title):
    fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)

    axes[0].plot(df['t_s'], rms_h, color='purple', label='RMS |AccH|')
    axes[0].axvspan(t_start, t_end, alpha=0.2, color='green', label='Selected window')
    axes[0].set_ylabel('RMS |AccH| (m/s²)')
    axes[0].set_title(f'{title} – Swing activity and window selection')
    axes[0].legend(); axes[0].grid(True, alpha=0.4)

    axes[1].plot(df['t_s'], rms_gz, color='red', label='RMS |GyrZ| (spin)')
    axes[1].axvspan(t_start, t_end, alpha=0.2, color='green')
    axes[1].axhline(20, color='gray', linestyle='--', linewidth=1, label='Spin threshold (20 °/s)')
    axes[1].set_ylabel('RMS |GyrZ| (deg/s)')
    axes[1].set_xlabel('Time (s)')
    axes[1].legend(); axes[1].grid(True, alpha=0.4)

    fig.tight_layout()
    plt.show()


plot_window_selection(df1, t1_start, t1_end, rms_h1, rms_gz1, 'Test 1')
plot_window_selection(df2, t2_start, t2_end, rms_h2, rms_gz2, 'Test 2')

## 5. Extract a Signed Oscillation Signal via PCA

Because the bucket rotates, the horizontal swing signal is spread across AccX and AccY.
We use **Principal Component Analysis (PCA)** on the two horizontal accelerations within the
selected window to find the dominant swing direction. Projecting onto that axis gives a
**signed** signal at the true pendulum frequency.

We also keep the horizontal magnitude for comparison — it oscillates at **twice** the pendulum
frequency, so we must halve any frequency we read from it.

In [ ]:
def extract_signals(df, t_start, t_end):
    """Slice the swinging window and compute the PCA-projected signal."""
    mask = (df['t_s'] >= t_start) & (df['t_s'] <= t_end)
    seg  = df[mask].copy()

    # Mean-subtract (remove the DC tilt bias) into new arrays — don't mutate .values
    ax = seg['AccX'].values - seg['AccX'].mean()
    ay = seg['AccY'].values - seg['AccY'].mean()

    # PCA via SVD on the 2-column data matrix
    X         = np.column_stack([ax, ay])
    _, sv, Vt = np.linalg.svd(X, full_matrices=False)
    pc1       = Vt[0]       # unit vector along dominant swing direction
    projected = X @ pc1     # signed oscillation signal

    seg['AccH_proj'] = projected

    variance_captured = sv[0]**2 / (sv**2).sum()
    angle_deg = np.degrees(np.arctan2(pc1[1], pc1[0]))
    print(f"  PCA swing direction: ({pc1[0]:.3f}, {pc1[1]:.3f})  "
          f"≈ {angle_deg:.1f}° from X-axis")
    print(f"  Variance in PC1: {variance_captured:.1%}")

    return seg, pc1


print("Test 1:")
seg1, pc1_1 = extract_signals(df1, t1_start, t1_end)
print("Test 2:")
seg2, pc1_2 = extract_signals(df2, t2_start, t2_end)

In [ ]:
def plot_pca_signals(seg, title):
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

    axes[0].plot(seg['t_s'], seg['AccH_proj'], color='steelblue', lw=0.8,
                 label='PCA-projected AccH (signed)')
    axes[0].axhline(0, color='k', lw=0.5)
    axes[0].set_ylabel('Acceleration (m/s²)')
    axes[0].set_title(f'{title} – PCA-projected horizontal acceleration (signed, DC-removed)')
    axes[0].legend(); axes[0].grid(True, alpha=0.4)

    axes[1].plot(seg['t_s'], seg['AccH_mag'], color='purple', lw=0.8,
                 label='|AccH| magnitude')
    axes[1].set_ylabel('|AccH| (m/s²)')
    axes[1].set_xlabel('Time (s)')
    axes[1].set_title('Horizontal magnitude (peaks at ~center of swing, frequency = 2 × pendulum frequency)')
    axes[1].legend(); axes[1].grid(True, alpha=0.4)

    fig.tight_layout()
    plt.show()


plot_pca_signals(seg1, 'Test 1')
plot_pca_signals(seg2, 'Test 2')

## 6. Frequency Analysis (FFT)

We apply an FFT to:
1. **PCA-projected signal** — dominant peak at the pendulum frequency $f_p$
2. **Horizontal magnitude** — dominant peak at $2f_p$

We also show the FFT of AccX and AccY individually.

In [ ]:
def compute_fft(signal_array, fs):
    """Return one-sided power spectrum (frequencies, power)."""
    n      = len(signal_array)
    window = np.hanning(n)
    sig_w  = (signal_array - signal_array.mean()) * window
    yf     = rfft(sig_w)
    xf     = rfftfreq(n, 1.0 / fs)
    power  = (2.0 / n * np.abs(yf))**2
    return xf, power


def fft_peak(xf, power, f_min=0.1, f_max=3.0):
    """Return the frequency of the strongest peak in [f_min, f_max] Hz."""
    mask = (xf >= f_min) & (xf <= f_max)
    idx  = np.argmax(power[mask])
    return xf[mask][idx]


def plot_fft(seg, fs, title):
    ax_mean  = seg['AccX'].values - seg['AccX'].mean()
    ay_mean  = seg['AccY'].values - seg['AccY'].mean()
    proj     = seg['AccH_proj'].values
    mag      = seg['AccH_mag'].values

    xf_p,  pw_p  = compute_fft(proj,    fs)
    xf_m,  pw_m  = compute_fft(mag,     fs)
    xf_ax, pw_ax = compute_fft(ax_mean, fs)
    xf_ay, pw_ay = compute_fft(ay_mean, fs)

    f_proj = fft_peak(xf_p,  pw_p)
    f_mag  = fft_peak(xf_m,  pw_m)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    for ax, xf, pw, lbl, col in zip(
            axes.flat,
            [xf_p,  xf_m,  xf_ax, xf_ay],
            [pw_p,  pw_m,  pw_ax, pw_ay],
            ['PCA-projected', '|AccH| magnitude', 'AccX', 'AccY'],
            ['steelblue', 'purple', 'C0', 'C1']):
        ax.plot(xf, pw, color=col, lw=0.9)
        ax.set_xlim(0, min(3, fs / 2))
        ax.set_xlabel('Frequency (Hz)')
        ax.set_ylabel('Power')
        ax.set_title(f'FFT – {lbl}')
        ax.grid(True, alpha=0.4)

    # Mark detected peaks
    axes[0, 0].axvline(f_proj, color='red',    linestyle='--',
                       label=f'Peak: {f_proj:.3f} Hz → T = {1/f_proj:.3f} s')
    axes[0, 1].axvline(f_mag,  color='red',    linestyle='--',
                       label=f'Peak: {f_mag:.3f} Hz → T = {2/f_mag:.3f} s (÷2)')
    axes[0, 0].legend(fontsize=9)
    axes[0, 1].legend(fontsize=9)

    fig.suptitle(f'{title} – Frequency spectra', fontsize=13)
    fig.tight_layout()
    plt.show()

    return f_proj, f_mag


print("Test 1 FFT:")
f1_proj, f1_mag = plot_fft(seg1, fs1, 'Test 1')
print(f"  PCA peak: {f1_proj:.4f} Hz,  |AccH| peak: {f1_mag:.4f} Hz")

print("\nTest 2 FFT:")
f2_proj, f2_mag = plot_fft(seg2, fs2, 'Test 2')
print(f"  PCA peak: {f2_proj:.4f} Hz,  |AccH| peak: {f2_mag:.4f} Hz")

## 7. Period Estimation via Peak Detection

As a cross-check we directly measure the time between successive maxima (half-cycles) in the
PCA-projected signal and in the horizontal magnitude.

In [ ]:
def estimate_period_peaks(t, sig, fs, label, prominence_frac=0.3):
    """Find peaks and estimate the oscillation period.

    Returns the mean period in seconds.
    """
    sig_centered = sig - sig.mean()
    prom = prominence_frac * (sig_centered.max() - sig_centered.min())
    min_dist = int(0.5 / np.median(np.diff(t)))  # at least 0.5 s apart

    peaks, props = signal.find_peaks(
        sig_centered, distance=min_dist, prominence=prom
    )

    if len(peaks) < 2:
        print(f"  {label}: too few peaks found ({len(peaks)}). Try adjusting thresholds.")
        return None, peaks

    peak_times = t[peaks]
    inter_peak = np.diff(peak_times)
    mean_inter = np.mean(inter_peak)
    std_inter  = np.std(inter_peak)

    print(f"  {label}: {len(peaks)} peaks found")
    print(f"    Inter-peak intervals: mean = {mean_inter:.4f} s,  std = {std_inter:.4f} s")
    return mean_inter, peaks


print("=== Test 1 ===")
t1 = seg1['t_s'].values

# PCA signal: peaks are one per full swing cycle → inter-peak ≈ T
T1_proj_inter, pk1_proj = estimate_period_peaks(
    t1, seg1['AccH_proj'].values, fs1, 'PCA-projected (full period from peaks)')

# Magnitude signal: peaks at every center-of-swing pass → inter-peak ≈ T/2
T1_mag_inter, pk1_mag = estimate_period_peaks(
    t1, seg1['AccH_mag'].values, fs1, '|AccH| magnitude (half period from peaks)')

print("\n=== Test 2 ===")
t2 = seg2['t_s'].values

T2_proj_inter, pk2_proj = estimate_period_peaks(
    t2, seg2['AccH_proj'].values, fs2, 'PCA-projected (full period from peaks)')

T2_mag_inter, pk2_mag = estimate_period_peaks(
    t2, seg2['AccH_mag'].values, fs2, '|AccH| magnitude (half period from peaks)')

In [ ]:
def plot_peaks(seg, fs, pk_proj, pk_mag, title):
    t    = seg['t_s'].values
    proj = seg['AccH_proj'].values
    proj_c = proj - proj.mean()
    mag  = seg['AccH_mag'].values
    mag_c = mag - mag.mean()

    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

    axes[0].plot(t, proj_c, color='steelblue', lw=0.8, label='PCA-projected (DC removed)')
    axes[0].plot(t[pk_proj], proj_c[pk_proj], 'rv', ms=8, label='Detected peaks')
    axes[0].axhline(0, color='k', lw=0.5)
    axes[0].set_ylabel('Acceleration (m/s²)')
    axes[0].set_title(f'{title} – Peak detection on PCA-projected signal')
    axes[0].legend(); axes[0].grid(True, alpha=0.4)

    axes[1].plot(t, mag_c, color='purple', lw=0.8, label='|AccH| magnitude (DC removed)')
    axes[1].plot(t[pk_mag], mag_c[pk_mag], 'rv', ms=8, label='Detected peaks')
    axes[1].set_ylabel('|AccH| (m/s²)')
    axes[1].set_xlabel('Time (s)')
    axes[1].set_title('Peak detection on horizontal magnitude')
    axes[1].legend(); axes[1].grid(True, alpha=0.4)

    fig.tight_layout()
    plt.show()


plot_peaks(seg1, fs1, pk1_proj, pk1_mag, 'Test 1')
plot_peaks(seg2, fs2, pk2_proj, pk2_mag, 'Test 2')

## 8. Consolidate Period Estimates

We combine the four independent estimates of the pendulum period $T$:

| Source | Period |
|--------|--------|
| FFT of PCA-projected signal | $T = 1/f_{\text{peak}}$ |
| FFT of horizontal magnitude | $T = 2/f_{\text{peak}}$ |
| Peak spacing in PCA signal  | $T \approx \overline{\Delta t_{\text{peaks}}}$ |
| Peak spacing in magnitude   | $T \approx 2\,\overline{\Delta t_{\text{peaks}}}$ |

In [ ]:
def collect_periods(f_proj, f_mag, T_proj_inter, T_mag_inter, label):
    estimates = {}

    if f_proj > 0:
        estimates['FFT (PCA-projected)']  = 1.0 / f_proj
    if f_mag > 0:
        estimates['FFT (|AccH| mag, ÷2)'] = 2.0 / f_mag
    if T_proj_inter is not None:
        estimates['Peak spacing (PCA)']   = T_proj_inter
    if T_mag_inter is not None:
        estimates['Peak spacing (|AccH|, ×2)'] = 2.0 * T_mag_inter

    print(f"\n{'─'*50}")
    print(f"  {label} – Period estimates")
    print(f"{'─'*50}")
    for name, T in estimates.items():
        print(f"  {name:<35s}  T = {T:.4f} s")

    T_values = np.array(list(estimates.values()))
    T_mean   = T_values.mean()
    T_std    = T_values.std()
    print(f"  {'─'*45}")
    print(f"  Mean period:  T = {T_mean:.4f} ± {T_std:.4f} s")
    return T_mean, T_std


T1_mean, T1_std = collect_periods(f1_proj, f1_mag, T1_proj_inter, T1_mag_inter, 'Test 1')
T2_mean, T2_std = collect_periods(f2_proj, f2_mag, T2_proj_inter, T2_mag_inter, 'Test 2')

## 9. Pendulum Length Estimation

For a simple pendulum (small-angle approximation):

$$T = 2\pi\sqrt{\frac{L}{g}} \quad\Longrightarrow\quad L = g\left(\frac{T}{2\pi}\right)^2$$

The uncertainty in $L$ from the uncertainty in $T$ propagates as:

$$\sigma_L = 2\,g\,\frac{T}{(2\pi)^2}\,\sigma_T = \frac{2\sigma_T}{T}\,L$$

In [ ]:
def estimate_length(T_mean, T_std, label):
    L_mean = g * (T_mean / (2 * np.pi))**2
    L_std  = g * 2 * T_mean * T_std / (2 * np.pi)**2   # error propagation

    print(f"\n{'═'*50}")
    print(f"  {label} – Pendulum length estimate")
    print(f"{'═'*50}")
    print(f"  Period:  T = {T_mean:.4f} ± {T_std:.4f} s")
    print(f"  Length:  L = {L_mean:.4f} ± {L_std:.4f} m")
    print(f"         L ≈ {L_mean*100:.1f} ± {L_std*100:.1f} cm")
    return L_mean, L_std


L1_mean, L1_std = estimate_length(T1_mean, T1_std, 'Test 1')
L2_mean, L2_std = estimate_length(T2_mean, T2_std, 'Test 2')

## 10. Summary and Comparison

In [ ]:
# -------------------------------------------------------------------
# Set this to the physically measured length of your pendulum (meters)
# Leave as None if you have not measured it yet.
# -------------------------------------------------------------------
L_measured_m = None   # e.g. L_measured_m = 1.05

labels = ['Test 1', 'Test 2']
T_vals = [T1_mean, T2_mean]
T_errs = [T1_std,  T2_std]
L_vals = [L1_mean, L2_mean]
L_errs = [L1_std,  L2_std]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Period comparison
ax = axes[0]
ax.bar(labels, T_vals, yerr=T_errs, capsize=8, color=['steelblue', 'seagreen'],
       alpha=0.8, edgecolor='k')
ax.set_ylabel('Period T (s)')
ax.set_title('Measured pendulum period')
ax.grid(True, axis='y', alpha=0.4)
for i, (T, Ts) in enumerate(zip(T_vals, T_errs)):
    ax.text(i, T + Ts + 0.01, f'{T:.3f} s', ha='center', fontsize=10)

# Length comparison
ax = axes[1]
ax.bar(labels, [v*100 for v in L_vals],
       yerr=[e*100 for e in L_errs], capsize=8,
       color=['steelblue', 'seagreen'], alpha=0.8, edgecolor='k')
if L_measured_m is not None:
    ax.axhline(L_measured_m * 100, color='red', linestyle='--', linewidth=2,
               label=f'Measured: {L_measured_m*100:.1f} cm')
    ax.legend()
ax.set_ylabel('Estimated length L (cm)')
ax.set_title('Estimated pendulum length  L = g(T/2π)²')
ax.grid(True, axis='y', alpha=0.4)
for i, (L, Ls) in enumerate(zip(L_vals, L_errs)):
    ax.text(i, L*100 + Ls*100 + 0.5, f'{L*100:.1f} cm', ha='center', fontsize=10)

fig.tight_layout()
plt.show()

# ---- Printed summary ----
print("\n" + "═"*60)
print(" SUMMARY")
print("═"*60)
for lbl, T, Ts, L, Ls in zip(labels, T_vals, T_errs, L_vals, L_errs):
    print(f"  {lbl}:  T = {T:.4f} ± {Ts:.4f} s  →  L = {L*100:.2f} ± {Ls*100:.2f} cm")

# Weighted average of the two tests
w1, w2 = 1/T1_std**2, 1/T2_std**2
T_wmean = (w1*T1_mean + w2*T2_mean) / (w1 + w2)
T_werr  = 1.0 / np.sqrt(w1 + w2)
L_wmean = g * (T_wmean / (2*np.pi))**2
L_werr  = g * 2*T_wmean*T_werr / (2*np.pi)**2
print()
print(f"  Weighted average:  T = {T_wmean:.4f} ± {T_werr:.4f} s")
print(f"  → L = {L_wmean*100:.2f} ± {L_werr*100:.2f} cm")

if L_measured_m is not None:
    diff = abs(L_wmean - L_measured_m)
    pct  = diff / L_measured_m * 100
    print()
    print(f"  Physically measured length: {L_measured_m*100:.1f} cm")
    print(f"  Discrepancy: {diff*100:.2f} cm  ({pct:.1f}%)")
print("═"*60)

## 11. Notes on Methodology

### Handling the rotating bucket
Because the bucket spins about the vertical axis (GyrZ ≠ 0), the pendulum swing direction rotates
between AccX and AccY. Two complementary strategies were applied:

1. **PCA projection** (§5): finds the principal oscillation direction in the AccX–AccY plane
   within the selected window. Works well when the rotation is slow enough that the swing
   direction doesn't rotate much within the window.

2. **Horizontal magnitude** (all sections): $|a_H| = \sqrt{\text{AccX}^2+\text{AccY}^2}$ is
   rotation-invariant but always positive, so its FFT peak appears at **twice** the pendulum
   frequency. We divide by 2 consistently.

### What the accelerometer measures
An IMU in the bucket reads the *specific force*: $\mathbf{f} = \mathbf{a}_{\text{inertial}} - \mathbf{g}$.
During the swing the dominant horizontal component oscillates at the pendulum frequency and is
proportional to $\sin\theta$ (gravity component projected onto horizontal) plus the centripetal
acceleration $L\dot{\theta}^2$. For small angles both terms oscillate near $f_p$.

### Small-angle assumption
The formula $L = g(T/2\pi)^2$ assumes small angles ($\theta_0 \lesssim 15°$). For larger amplitudes
the true period is slightly longer:
$$T \approx 2\pi\sqrt{\frac{L}{g}}\left(1 + \frac{\theta_0^2}{16}\right)$$
If the initial swing amplitude was large, the raw IMU data overestimates $T$ slightly, leading
to a slight overestimate of $L$. The accelerometer amplitude at the extremes can be used to
estimate $\theta_0$ for a first-order correction if needed.